# Dimension Reduction

## Stochastic Neighbor Embedding (SNE)

The objective of this method is to find an appropriate representation of a set of $n$ data points in dimension $d$ of space $(x_1, x_2, \cdots, x_n) \in \mathbb{R}^{n \times d}$ into a space of dimension $d' \leq d$. This resulting data set is noted $(y_1, y_2, \cdots, y_n) \in \mathbb{R}^{n \times d'}$.

The key idea of t-SNE is to conserve distance between points, independantly of their precise configuration. First, define the conditional probability that particle $j$ is at a given distance of particle $i$:
$$
p_{j|i} = \frac{e^{-||x_i - x_j||^2/(2\sigma_i^2)}}{\sum\limits_{k \neq i}e^{-||x_i - x_k||^2/(2\sigma_i^2)}},
$$
meaning that each particle is the center of a gaussian law of variance $\sigma^2_i$ describing the probability density of its neighbouring points.
Very similarly, now define:
$$
q_{j|i} = \frac{e^{-||y_i - y_j||^2}}{\sum\limits_{k \neq i}e^{-||y_i - y_k||^2}}.
$$

Note that the distance appearing in $p_{j|i}$ are in $\mathbb{R}^d$ whereas the distance is in $\mathbb{R}^{d'}$ for $q_{i|j}$.
We set $q_{i|i}=p_{i|i}=0$.

The goal is then to find the points $(y_1, y_2, \cdots, y_n) \in \mathbb{R}^{n \times d'}$ that minimizes the discrepency between $q_{i|j}$ and $p_{i|j}$. We take here a strong metric, namely, the Kullback-Leibler divergence:
$$
C = KL(P||Q) = \sum\limits_{i=1}^n KL(P_i||Q_i) = \sum\limits_{i=1}^n \sum_{j = 1}^n p_{j|i} \log(\frac{p_{j|i}}{q_{j|i}}) = \sum\limits_{i=1}^n H(P_i||Q_i) - H(P_i),
$$
assuming $\frac{p_{i|i}}{q_{i|i}} = 1$. Note that $Q$ is the distribution on the right side of the divergence, meaning that it is equivalent to a Maximum likelihood objective. The optimal point will be find via a first order steepest ascent method. Keeping only terms with $Q_i$ in the KL divergence, one obtains:
$$
KL(P_i, Q_i) \propto -\sum_{j=1}^n p_{j|i}\log(q_{j|i}) = \sum_{j=1}^n p_{j|i} \bigg(||y_i - y_j||^2 + \log\big(\sum\limits_{k\neq i} e^{-||y_k - y_i||^2}\big) \bigg)
$$

A major weakness appears here, the objective function is not convex w.r.t $(y_1, \cdots, y_n)$.

The main remaining question is: how to choose the 'best' $\sigma_i$ for each particle $i$. A small $\sigma_i$ is suited for dense regions, while large $\sigma_i$ is very much relevant for sparse regions.

With this objective in mind, one can define the perplexity of $p_i = p_{\cdot|j}$ as $\text{Perp}(p_i) = 2^{H(p_i)}$, where the entropy $H$ of $p_i$ is $H(p_i) = -\sum\limits_{j=1}^n p_{j|i} \log(p_{j|i})$. The operator fixes a perplexity target $P_i$, one per particle $X_i$ in the data set, and SNE performs a binary search for the value of $\sigma_i$ that produce this value. It can be done with a binary search precisely because the entropy is an increasing function of $\sigma_i$, one can convince ourself easily with a little bit of physical intuition by interpreting $\sigma_i$ as a temperature. Typical value of particle perplexity should be between $5$ and $50$. Perplexity is a smooth effective measure of the number of neighbours.

To perform gradient ascent, the gradient of the Kullback-Leibler divergence w.r.t $Y_i$ should be computed. It admits a surprising simple expression:
$$
\frac{\partial C}{\partial y_i} = 2\sum\limits_{j=1}^n (p_{j|i} - q_{j|i} + p_{i|j} - q_{i|j})(y_i - y_j)
$$

An updates takes the form, $\forall i \in \{1, \cdots, d'\}$:
$$
y_{i}^{(n+1)} =y_i^{(n)} + \eta_n \frac{\partial C}{\partial y_i} + \alpha_n(y_i^{(n)} - y_i^{(n-1)}),
$$
where $\eta_n$ is the learning rate and $\alpha_n$ is the momentum parameter at iteration $n$.

During the first iterations, a gaussian noise with decreasing variance is added.

In a nutshell, I should make a method to compute $p_{i|j}$, which amounts to compute the distance $||x_i - x_j||_2^2$ for all pairs of points. I should make another method to figure out an appropriate $\sigma_i$ for each of the points. Eventually, a method to run the optimization algorithm is necessary.

[1] https://www.jmlr.org/papers/volume9/vandermaaten08a/vandermaaten08a.pdf

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# def compute_kern_dist(X: np.ndarray, kern: callable):
#     return kern(X, X)

def compute_eucl_dist(X: np.ndarray):
    """
    Compute the euclidian distance between all pairs of points.
    """
    return np.sum(X**2, axis=-1)

def compute_pji(X: np.ndarray, sigma: np.ndarray):
    """
    Compute the distribution matrix whose i-th row and j-th column is the probability of having particle j at this distance from particle i.
    Args:
        - X (np.ndarray): Set of data point. It should be of shape (n x d), where n is the number of samples, and d the dimension of space they live in.
        - sigma (np.ndarray): Variange of the gaussian. It should be of shape (n,).
    """
    assert np.all(sigma > 0), "All entry in sigma array should be positive."
    dist = compute_eucl_dist(X[None,...] - X[:,None,...]) #Compute the distance between all pairs of points.
    P = np.exp(-dist/sigma[:, None])
    idx = np.concatenate((np.arange(len(X))[...,None], np.arange(len(X))[..., None]), axis=-1)
    P[idx] = 0
    P /= (np.sum(P, axis=-1))[:,None]
    return P

def compute_qji(Y: np.ndarray):
    return compute_pji(Y, sigma=np.ones(len(Y)))

def compute_entropy(P: np.ndarray):
    assert np.sum(P) == 1, "P should be a list of probabilities."
    return -np.sum(P*np.log(P), axis = -1)

def sample_init(n: int, d: int, std: float=1e-2):
    """
    Initialize a set of points (y_1, ..., y_n) in the low dimensional space. It is done by sampling from a random variable.
    Args:
        std: standard deviation for the sampling.
        n: number of samples.
        d: dimension space.
    Returns:
        An array of shape (n, d).
    """
    assert std > 0, "The standard deviation should be positive."

    return np.random.randn(n, d)*std

def compute_grad_C(y: np.ndarray, matrix_pji: np.ndarray[np.ndarray], matrix_qji: np.ndarray[np.ndarray]):
    grad_C = np.empty(len(y))
    
    for i in range(len(y)):
        vect_Y = y[i] - y
        p = matrix_pji[i] - matrix_qji[i]
        p += matrix_pji[:,i] - matrix_qji[:,i]
        grad_C[i] = 2*np.sum(p*vect_Y)
    return grad_C

def find_sigma(X: np.ndarray, Perp_tar: np.ndarray):
    """Find the variance for a given perplexity target

    Args:
        X (np.ndarray): Dataset. Should be of shape (n x d). n is the number of samples and d the dimension of space.
        Perp (np.ndarray): Target perplexity for each points in the dataset.
    """
    assert len(X) == len(Perp_tar), "X and Perp_tar should be the same length"
    Sigma_BS = []

    for i, x, perp in enumerate(zip(X, Perp_tar)):
        entropy = np.infty
        sigma_min = 1e-2
        sigma_max = 30
        sigma = 15
        while np.abs(np.power(2, entropy) - perp) > 1e-3:
            dist = compute_eucl_dist(x - X) #Compute the distance between all pairs of points.
            vect = np.exp(-dist/sigma)
            vect[i] = 0
            vect /= np.sum(vect)
            entropy = compute_entropy(vect)
            if np.power(2, entropy) > perp:
                sigma_max = sigma
                sigma = np.sqrt(sigma*sigma_min)
            else:
                sigma_min = sigma
                sigma = np.sqrt(sigma*sigma_max)
            Sigma_BS.append(sigma)


if __name__ == "__main__":
    A = np.array([[2, 1], [0, 0], [5, 5]])
    sigma = np.array([1, 10, 2])
    compute_pji(A, sigma)

array([[0.00000000e+00, 9.99999998e-01, 2.06115362e-09],
       [9.89013057e-01, 0.00000000e+00, 1.09869426e-02],
       [9.99996273e-01, 3.72663928e-06, 0.00000000e+00]])

## t-distributed Stochastinc Neighbor Embedding (t-SNE)

While it is very similary to SNE, it possess two major differences. The first one is the use of symmetrized cost with simpler gradients, and the second one is that probability density are represented by a t-distribution, and not a gaussian distribution anymore. Symmetrized probability distribution are more robust to outlier

One introduces the symmetrized probability:
$$
p_{ij} = \frac{p_{i|j} + p_{j|i}}{2n}.
$$

One then introduces the symmetrized cost:
$$
KL(P||Q) = \sum\limits_{i=1}^n \sum\limits_{j=1}^n p_{ij} \log(\frac{p_{ij}}{q_{ij}}),
$$
with as before $p_{ii} = q_{ii} = 0$.

The second changement is the ansatz on $q_{ij}$. The unkowns are still the positions $(y_1, \cdots, y_n) \in \mathbb{R}^{n\times d'}$, but this time, the density family is the student distribution $\forall i\neq j$:
$$
q_{ij} = \frac{(1 + ||y_i - y_j||^2)^{-1}}{\sum\limits_{k=1}^n \sum\limits_{l \neq k} (1 + ||y_l - y_k||^2)^{-1}},
$$
so that the renormalization constant implies $\sum\limits_{i, j} q_{ij} = 1$.

The gradient simply becomes:
$$
\frac{\partial C}{\partial y_i} = 4 \sum_{j=1}^n(p_{ij} - q_{ij})(y_i - y_j)(1 + ||y_i - y_j||^2)^{-1},
$$
and the optimization scheme is the same as above.

The optimization of the t-SNE objective function is much easier than the cost function of SNE. A fixed number of $T = 1000$ timestep is fixed. The momentum term should be $\alpha(t) = .5$ if $t < 250$ and $\alpha(t) = .8$ if $t \geq 250$.


In [ ]:
def compute_symm_pij(X: np.ndarray, sigma):
    """
    Compute the symmetric probability distribution.
    """
    matrix = compute_pji(X, sigma)
    matrix += matrix.T
    matrix /= 1/(2*len(X))
    return matrix

def compute_symm_qij(Y: np.ndarray):
    """
    Compute student's distribution
    """
    Q = 1/(1 + compute_eucl_dist(Y[None,...] - Y[:,None,...]))
    idx = np.concatenate((np.arange(len(Y))[...,None], np.arange(len(Y))[..., None]), axis=-1)
    Q[idx] = 0 #Cancel the diagonal terms
    Q /= np.sum(Q)
    return Q

def compute_grad_C(y: np.ndarray, matrix_pij: np.ndarray[np.ndarray], matrix_qij: np.ndarray[np.ndarray]):
    grad_C = np.empty(len(y))
    for i in range(len(y)):
        p = matrix_pij[i]
        q = matrix_qij[i]
        dist = 1/(1 + np.sum((y[i] - y)**2, axis = -1))
        grad_C[i] = 4*np.sum((p-q)*(y[i] - y)*dist)
    return grad_C